# COLAB MEND (GPT-2) FT-Compare R1/R2

Run MEND R1 and R2 with your FT-comparable setup.


## Runtime
- Python 3
- T4 GPU
- If dependency conflict occurs: Runtime -> Restart runtime


In [ ]:
# Purpose: Mount Drive and prepare workspace.
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/work/results/MEND', exist_ok=True)
print('workspace ready')


In [ ]:
# Purpose: Clone EasyEdit and install dependencies.
%cd /content
!rm -rf /content/EasyEdit
!git clone --depth 1 https://github.com/zjunlp/EasyEdit.git

!pip -q install -U pip setuptools wheel
!pip -q install "PyYAML==6.0.2"
!pip -q install -r /content/EasyEdit/requirements.txt
!pip -q install "transformers==4.57.1" "tokenizers==0.22.1" "sentence-transformers==3.2.1"
!pip -q install "hydra-core" "omegaconf" "einops" "higher" "iopath" "fairscale" "timm" "peft"
!pip -q install "qwen_vl_utils" "zhipuai" "rouge" "av" "pyjwt==2.8.0"


In [ ]:
# Purpose: Copy project scripts/data from Drive.
import os, shutil
CANDIDATES = ['/content/drive/MyDrive/????LLM', '/content/drive/MyDrive/LLM']
ROOT = next((c for c in CANDIDATES if os.path.exists(c)), None)
assert ROOT is not None, 'Project root not found. Edit CANDIDATES.'
print('PROJECT ROOT =', ROOT)

%cd /content/work
!rm -rf /content/work/*

for fn in ['scripts/run_edit_fairness_rounds.py','scripts/run_fairness_eval.py','scripts/calc_drift_metrics.py','scripts/plot_drift.py']:
    src = os.path.join(ROOT, fn)
    assert os.path.exists(src), src
    shutil.copy(src, '/content/work')

for fn in ['data/edits_bias_stress_60.json','data/crows_pairs_sample.jsonl','data/bbq_sample.jsonl']:
    src = os.path.join(ROOT, fn)
    assert os.path.exists(src), src
    shutil.copy(src, '/content/work')

print('copied ok')


In [ ]:
# Purpose: Create MEND hparams file (replace checkpoint path first).
MEND_CKPT = '/content/drive/MyDrive/REPLACE_WITH_YOUR_MEND_CHECKPOINT.pt'  # TODO

hparams = f"""archive: "{MEND_CKPT}"
alg_name: "MEND"
device: 0
model_name: "gpt2"

model_class: GPT2LMHeadModel
tokenizer_class: GPT2Tokenizer
tokenizer_name: "gpt2"
model_parallel: false
inner_params:
  - transformer.h.9.mlp.c_proj.weight
  - transformer.h.9.mlp.c_fc.weight
  - transformer.h.10.mlp.c_proj.weight
  - transformer.h.10.mlp.c_fc.weight
  - transformer.h.11.mlp.c_proj.weight
  - transformer.h.11.mlp.c_fc.weight

alg: MEND
lr: 1e-6
edit_lr: 1e-4
lr_lr: 1e-4
lr_scale: 1.0
seed: 42
cedit: 0.1
cloc: 1.0
cbase: 1.0
dropout: 0.0
train_base: false
no_grad_layers: null
one_sided: false
n_hidden: 1
hidden_dim: null
init: id
norm: true
combine: true
x_only: false
delta_only: false
act: relu
rank: 768
mlp_class: IDMLP
shared: true

batch_size: 1
model_save_pt: 5000
silent: false
max_iters: 100000
log_interval: 1000
eval_log_interval: 1000
final_eval: true
val_interval: 1000
early_stop_patience: 20000
early_stop_key: "loss/total_edit_val"
eval_only: true
half: false
debug: false
save: false
verbose: true

val_batch_size: 5
accumulate_bs: 10
val_steps: 500
opt: Adam
grad_clip: 100.0

results_dir: ./results
"""

open('/content/work/MEND_gpt2_ft_compare.yaml', 'w', encoding='utf-8').write(hparams)
import os
print('saved /content/work/MEND_gpt2_ft_compare.yaml')
print('ckpt exists =', os.path.exists(MEND_CKPT), MEND_CKPT)


In [ ]:
# Purpose: Preflight check.
import os, importlib.util as iu, torch
mods=['torch','transformers','sentence_transformers','yaml','hydra','omegaconf','higher','iopath','fairscale','timm','peft','qwen_vl_utils','zhipuai','rouge','av']
print('cuda=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu=', torch.cuda.get_device_name(0))
print('missing=', [m for m in mods if iu.find_spec(m) is None])
for p in ['/content/work/run_edit_fairness_rounds.py','/content/work/run_fairness_eval.py','/content/work/calc_drift_metrics.py','/content/work/plot_drift.py','/content/work/MEND_gpt2_ft_compare.yaml','/content/work/edits_bias_stress_60.json','/content/work/crows_pairs_sample.jsonl','/content/work/bbq_sample.jsonl']:
    print(p, os.path.exists(p))


In [ ]:
# Purpose: Run MEND R1 (10 rounds, step=6).
import os, subprocess
OUT_DIR='/content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/rounds'
os.makedirs(OUT_DIR, exist_ok=True)
cmd=['python','-u','run_edit_fairness_rounds.py','--hparams','/content/work/MEND_gpt2_ft_compare.yaml','--edits_json','/content/work/edits_bias_stress_60.json','--crows','/content/work/crows_pairs_sample.jsonl','--bbq','/content/work/bbq_sample.jsonl','--rounds','10','--step','6','--start_round','0','--out_dir',OUT_DIR]
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TOKENIZERS_PARALLELISM']='false'; env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True,max_split_size_mb:64'; env['PYTHONPATH']='/content/EasyEdit:'+env.get('PYTHONPATH','')
p=subprocess.Popen(cmd,cwd='/content/work',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='')
ret=p.wait()
if ret!=0:
    raise RuntimeError(f'R1 failed: {ret}')


In [ ]:
# Purpose: Compute/plot/download R1 outputs.
!python /content/work/calc_drift_metrics.py --input_dir /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/rounds --pattern "round_*.json" --out /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/drift_metrics.json
!python /content/work/plot_drift.py --metrics_json /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/drift_metrics.json --out_png /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/drift_curve.png

%cd /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1
!zip -r mend_gpt2_ft_compare_r1.zip rounds drift_metrics.json drift_curve.png
from google.colab import files
files.download('/content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R1/mend_gpt2_ft_compare_r1.zip')


In [ ]:
# Purpose: Run MEND R2 (same config, independent repeat).
import os, subprocess
OUT_DIR='/content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/rounds'
os.makedirs(OUT_DIR, exist_ok=True)
cmd=['python','-u','run_edit_fairness_rounds.py','--hparams','/content/work/MEND_gpt2_ft_compare.yaml','--edits_json','/content/work/edits_bias_stress_60.json','--crows','/content/work/crows_pairs_sample.jsonl','--bbq','/content/work/bbq_sample.jsonl','--rounds','10','--step','6','--start_round','0','--out_dir',OUT_DIR]
env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TOKENIZERS_PARALLELISM']='false'; env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True,max_split_size_mb:64'; env['PYTHONPATH']='/content/EasyEdit:'+env.get('PYTHONPATH','')
p=subprocess.Popen(cmd,cwd='/content/work',env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in p.stdout:
    print(line,end='')
ret=p.wait()
if ret!=0:
    raise RuntimeError(f'R2 failed: {ret}')


In [ ]:
# Purpose: Compute/plot/download R2 outputs.
!python /content/work/calc_drift_metrics.py --input_dir /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/rounds --pattern "round_*.json" --out /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/drift_metrics.json
!python /content/work/plot_drift.py --metrics_json /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/drift_metrics.json --out_png /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/drift_curve.png

%cd /content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2
!zip -r mend_gpt2_ft_compare_r2.zip rounds drift_metrics.json drift_curve.png
from google.colab import files
files.download('/content/work/results/MEND/MEND_gpt2_ft_compare_10rounds_R2/mend_gpt2_ft_compare_r2.zip')
